#  Milestone 2 Exploratory Data Analysis & Statistical Reasoning

1. **AI model adoption** Which industries, developer roles, and countries show the highest AI model adoption, and which specific AI models are favored where?
2. **AI-learning path** Does how developers learn to use AI-assisted coding tools (curiosity-driven vs. job-required) differ by years of coding experience or developer role, and does that relate to what makes them endorse a tool to colleagues?

## 0. Setup and load

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.max_columns', 50)
df = pd.read_csv('survey_clean.csv')
df.shape

(49191, 34)

# Derived variables

Two variables the raw columns don't give us directly, both needed for the comparisons below.
**AI-learning path** collapse `LearnCodeAI`'s five verbose categories into the three groups the research question actually compares: curiosity-driven, job-required, and did-not-learn-AI. This also folds in respondents who report "no time spent learning" into the same "no AI learning" bucket as those who learned something unrelated to AI.

In [2]:
def ai_path(val):
    if pd.isna(val):
        return np.nan
    elif 'personal curiosity' in val:
        return 'Curiosity-driven'
    elif 'required for my job' in val:
        return 'Job-required'
    else:
    				return 'No AI learning'

df['AI_learn_path'] = df['LearnCodeAI'].apply(ai_path)
df['AI_learn_path'].value_counts(dropna=False)

AI_learn_path
Job-required        21028
Curiosity-driven    19309
No AI learning       4864
NaN                  3990
Name: count, dtype: int64

**Experience tier** bucket `YearsCode_num` into three groups so we can compare AI-learning path across career stage without every individual year being its own category. Boundaries follow common early/mid/senior-career conventions (0–5, 6–15, 16+ years) rather than splitting at the sample's own quartiles, so the groups mean the same thing to a reader outside this dataset.

In [8]:
def experience_tier(years):
    if pd.isna(years):
        return np.nan
    elif years <= 5:
        return '0-5 yrs (early career)'
    elif years <= 15:
        return '6-15 yrs (mid career)'
    else:
    				return '16+ yrs (senior)'

df['experience_tier'] = df['YearsCode_num'].apply(experience_tier)
df['experience_tier'].value_counts(dropna=False)

experience_tier
16+ yrs (senior)          18176
6-15 yrs (mid career)     18006
0-5 yrs (early career)     6860
NaN                        6149
Name: count, dtype: int64

### Years of coding experience (shape, center, spread)

In [9]:
df['YearsCode_num'].describe()

count    43042.000000
mean        16.570861
std         11.787610
min          1.000000
25%          8.000000
50%         14.000000
75%         24.000000
max        100.000000
Name: YearsCode_num, dtype: float64

In [10]:
# Text histogram-equivalent: distribution across the same tiers used for the comparison below
df['experience_tier'].value_counts(normalize=True).round(3)

experience_tier
16+ yrs (senior)          0.422
6-15 yrs (mid career)     0.418
0-5 yrs (early career)    0.159
Name: proportion, dtype: float64

**Reading:** experience is right-skewed (mean ~16.6 years pulled above the median ~14 by a long tail toward 50+), so the median is the more representative single number, and the tiering above is preferable to a single mean when comparing groups.

### AI-learning path

In [10]:
df['AI_learn_path'].value_counts(normalize=True).round(3)

AI_learn_path
Job-required        0.465
Curiosity-driven    0.427
No AI learning      0.108
Name: proportion, dtype: float64

**Reading:** among respondents who answered this question, more report AI learning tied to their job (job-required) than out of curiosity, and a meaningful minority did not engage with AI learning at all in the past year.

### Developer role (top categories)

In [12]:
df['DevType'].value_counts(normalize=True).head(8).round(3)

DevType
Developer, full-stack                            0.283
Developer, back-end                              0.148
Student                                          0.069
Architect, software or solutions                 0.061
Developer, front-end                             0.045
Developer, desktop or enterprise applications    0.044
Other (please specify):                          0.042
Developer, mobile                                0.032
Name: proportion, dtype: float64

## AI-learning path by experience tier

The core comparison the research question asks for.

In [13]:
learn_by_exp = pd.crosstab(df['experience_tier'], df['AI_learn_path'], normalize='index')
learn_by_exp = learn_by_exp[['Curiosity-driven', 'Job-required', 'No AI learning']]
learn_by_exp.round(3)

AI_learn_path,Curiosity-driven,Job-required,No AI learning
experience_tier,,,
0-5 yrs (early career),0.467,0.419,0.114
16+ yrs (senior),0.408,0.480,0.112
6-15 yrs (mid career),0.434,0.466,0.100


**Reading:** curiosity-driven learning is highest among early-career respondents (46.7%) and steadily declines through mid-career (43.4%) to senior (40.8%), while job-required learning moves the opposite direction, rising from 41.9% to 48.0% consistent with a story where early-career developers explore AI tools on their own time, while senior developers more often adopt them because their job now expects it. The "no AI learning" share does not follow a clean trend with seniority (11.4% early, 10.0% mid, 11.1% senior) it is roughly flat, not growing so that third category should be read as background noise rather than part of the main story.

In [14]:
# Raw counts behind the proportions above, since normalized crosstabs can hide small groups
pd.crosstab(df['experience_tier'], df['AI_learn_path'])

AI_learn_path,Curiosity-driven,Job-required,No AI learning
experience_tier,,,
0-5 yrs (early career),3189,2863,776
16+ yrs (senior),7413,8705,2031
6-15 yrs (mid career),7800,8368,1800


## AI-learning path by developer role

Restricted to the six largest `DevType` categories so the comparison isn't dominated by roles with only a handful of respondents.

In [15]:
top_roles = df['DevType'].value_counts().head(6).index
role_subset = df[df['DevType'].isin(top_roles)]
learn_by_role = pd.crosstab(role_subset['DevType'], role_subset['AI_learn_path'], normalize='index')
learn_by_role = learn_by_role[['Curiosity-driven', 'Job-required', 'No AI learning']]
learn_by_role.round(3)

AI_learn_path,Curiosity-driven,Job-required,No AI learning
DevType,,,
"Architect, software or solutions",0.397,0.530,0.073
"Developer, back-end",0.409,0.478,0.113
"Developer, desktop or enterprise applications",0.423,0.421,0.156
"Developer, front-end",0.394,0.496,0.110
"Developer, full-stack",0.406,0.498,0.096
Student,0.607,0.251,0.142


**Reading:** most of these roles look similar to each other and to the overall split but Students stand out sharply, at 60.7% curiosity-driven versus roughly 39–41% for every professional developer role shown. That single category, not a smooth gradient across roles, is what drives the spread in this table; once Students are set aside, role explains noticeably less of the variation than experience tier does in Section 3. This makes sense structurally students are less likely to have a job dictating their tool use and is worth calling out explicitly rather than reading it as "developer role predicts AI-learning motivation" in general.

### Segment comparison does AI-learning path relate to tool-endorsement rankings?

The second half of the research question: among respondents who *did* endorse a tool (so `TechEndorse_1`–`_8` are populated), does the top-ranked endorsement factor differ between curiosity-driven and job-required learners? Lower numbers here indicate higher rank (1 = most important), per the original survey's scale.

In [11]:
endorse_cols = [f'TechEndorse_{i}' for i in range(1, 9)]
endorsed = df.dropna(subset=endorse_cols)
endorsed.shape

(35975, 36)

In [12]:
avg_rank_by_path = endorsed.groupby('AI_learn_path')[endorse_cols].mean().round(2)
avg_rank_by_path

,TechEndorse_1,TechEndorse_2,TechEndorse_3,TechEndorse_4,TechEndorse_5,TechEndorse_6,TechEndorse_7,TechEndorse_8
AI_learn_path,,,,,,,,
Curiosity-driven,8.17,4.16,4.15,5.70,4.21,5.10,6.58,4.48
Job-required,7.47,4.07,4.07,5.69,4.04,5.37,6.41,4.26
No AI learning,8.39,4.02,4.13,5.54,4.11,5.12,6.35,4.41


**Reading:** compare rows a lower average in a given `TechEndorse_n` column means that endorsement factor was ranked more important, on average, by that group. Any column where curiosity-driven and job-required rows diverge by close to a full rank point is a candidate "what's different about how these two groups decide to endorse a tool" the raw survey codebook (not included in this dataset export) would be needed to label what each `TechEndorse_n` factor represents; that mapping is listed as a limitation in the report rather than guessed at here.

## Outlier and anomaly review

Section 5.1 of the cleaning notebook flagged 248 respondents (about 0.5%) reporting more than 50 years of coding experience via `YearsCode_outlier_flag`. Rather than silently including or excluding them, we check whether Section 3's main finding holds with that tail removed.

In [18]:
df['YearsCode_outlier_flag'].sum()

np.int64(248)

In [17]:
no_outliers = df[~df['YearsCode_outlier_flag'].fillna(False)]
learn_by_exp_robust = pd.crosstab(no_outliers['experience_tier'], no_outliers['AI_learn_path'], normalize='index')
learn_by_exp_robust = learn_by_exp_robust[['Curiosity-driven', 'Job-required', 'No AI learning']]
learn_by_exp_robust.round(3)

AI_learn_path,Curiosity-driven,Job-required,No AI learning
experience_tier,,,
0-5 yrs (early career),0.467,0.419,0.114
16+ yrs (senior),0.407,0.482,0.111
6-15 yrs (mid career),0.434,0.466,0.100


**Reading:** compare against Section 3's table the pattern (curiosity declines, job-required rises, with experience) is unchanged after removing the 248 outlier rows, since they are a tiny fraction of the "16+ yrs" tier specifically. The finding is not an artifact of a handful of extreme responses.

### Statistical reasoning: is the experience/AI-learning relationship likely real, or could it be sampling noise?

A chi-square test of independence on the raw counts (not the outlier-filtered version, to keep the test on the full available sample) testing whether AI-learning path and experience tier are independent.

In [20]:
contingency = pd.crosstab(df['experience_tier'], df['AI_learn_path'])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"chi2 = {chi2:.1f}, degrees of freedom = {dof}, p-value = {p_value:.2e}")

chi2 = 95.7, degrees of freedom = 4, p-value = 7.98e-20


**Plain-language interpretation:** the p-value here is far below any conventional 0.05 threshold, meaning it is very unlikely we'd see a relationship this strong between experience tier and AI-learning path if the two were truly unrelated in the population this survey draws from. **This does not mean experience *causes* a particular AI-learning path** a third factor (e.g. seniority correlating with management responsibilities that come with job-mandated tooling) could explain both. It also does not tell us the relationship is large or practically important with a sample this size (tens of thousands of respondents), even small, practically trivial differences can produce a very low p-value.

In [18]:
# A 95% confidence interval on one specific, question-relevant proportion:
# share of early-career respondents whose AI learning was curiosity-driven
early = df[df['experience_tier'] == '0-5 yrs (early career)']['AI_learn_path'].dropna()
p_hat = (early == 'Curiosity-driven').mean()
n = len(early)
se = np.sqrt(p_hat * (1 - p_hat) / n)
ci_low, ci_high = p_hat - 1.96 * se, p_hat + 1.96 * se
print(f"n = {n}, proportion curiosity-driven = {p_hat:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")

n = 6828, proportion curiosity-driven = 0.467
95% CI: [0.455, 0.479]


**Plain-language interpretation:** we're roughly 95% confident the true share of early-career respondents (in the population this survey represents) who learned AI tools out of curiosity falls within this interval. Because `n` is large, the interval is narrow the estimate is precise, though again, "precise" describes the *survey respondents*, not developers generally.

Pitfall check (explicit, per the rubric)

- **Misleading axes / scale:** not applicable to the tables above; any bar chart built from these proportions in the visualization set will start its y-axis at 0.
- **Correlation vs. causation:** the chi-square result shows association, not that experience causes the AI-learning path.
- **Cherry-picking supportive subsets:** tested the main finding against the full sample and the outlier-filtered sample rather than reporting only whichever version looked cleaner.
- **Small-sample overconfidence:** the experience-tier comparison  and the chi-square test both draw on thousands of respondents per group; the endorsement-ranking comparison, draws on a smaller, self-selected subgroup of respondents who reported actually endorsing a tool — that smaller, non-random subgroup is named as a limitation in the report, not treated with the same confidence finding.
- **Sampling considerations:** this is a self-selected survey of respondents who chose to participate, not a random sample of all developers. Every proportion and p-value above describes this respondent pool; the report's limitations section restates this rather than letting the statistics imply a broader population claim.

## Group question — AI model adoption by industry, role, and country

The group shifted the shared research direction to **AI model adoption**: which industries, developer roles, and countries show the highest adoption of AI models/tools (`AIModelsHaveWorkedWith` in the cleaned data, summarized as the boolean `AI_adopter`), and which specific models are favored where.

### Overall adoption rate

In [22]:
df['AI_adopter'].value_counts(normalize=True).round(3)

AI_adopter
False    0.669
True     0.331
Name: proportion, dtype: float64

**Reading:** a little over a third of respondents report having worked with at least one AI model this baseline rate is the yardstick every industry/role/country breakdown below should be compared against, rather than reading any single group's raw percentage in isolation.

## Adoption by industry

In [23]:
industry_adopt = df.groupby('Industry')['AI_adopter'].agg(['mean', 'count']).rename(columns={'mean': 'adoption_rate', 'count': 'n'})
industry_adopt = industry_adopt[industry_adopt['n'] >= 50].sort_values('adoption_rate', ascending=False)
industry_adopt.round(3)

,adoption_rate,n
Industry,,
Fintech,0.466,1690
Media & Advertising Services,0.463,773
Software Development,0.451,16282
Retail and Consumer Services,0.443,1053
"Internet, Telecomm or Information Services",0.438,1698
Insurance,0.438,434
Healthcare,0.429,1397
Other:,0.428,2667
Banking/Financial Services,0.423,1437


**Reading:** industries are filtered to those with at least 50 respondents so a tiny group doesn't produce a misleadingly extreme rate. Compare each industry's `adoption_rate` against the ~9.1.1 baseline industries clustered near the baseline are not meaningfully different from the overall population; only ones clearly above or below it are worth calling out in the report.

## Adoption by developer role (DevType)

In [24]:
role_adopt = df.groupby('DevType')['AI_adopter'].agg(['mean', 'count']).rename(columns={'mean': 'adoption_rate', 'count': 'n'})
role_adopt = role_adopt[role_adopt['n'] >= 50].sort_values('adoption_rate', ascending=False)
role_adopt.round(3)

,adoption_rate,n
DevType,,
"Senior executive (C-suite, VP, etc.)",0.553,528
"Founder, technology or otherwise",0.515,431
AI/ML engineer,0.501,677
Engineering manager,0.463,1068
Data scientist,0.456,574
"Developer, AI apps or physical AI",0.449,236
"Architect, software or solutions",0.420,2684
Data engineer,0.410,770
"Developer, full-stack",0.400,12351


**Reading:** same 50-respondent floor applied. Roles most exposed to AI tooling day-to-day (e.g. AI/ML-adjacent roles) should sit well above the baseline; roles with little day-to-day coding exposure should sit below it this is a useful check that the numbers make substantive sense, not just a ranking exercise.

## Adoption by country, and which models are favored

Restricted to countries with a reasonable respondent count, since a country with 3 total responses produces an unstable rate.

In [25]:
country_adopt = df.groupby('Country')['AI_adopter'].agg(['mean', 'count']).rename(columns={'mean': 'adoption_rate', 'count': 'n'})
country_adopt = country_adopt[country_adopt['n'] >= 100].sort_values('adoption_rate', ascending=False)
country_adopt.round(3).head(15)

,adoption_rate,n
Country,,
China,0.616,255
Taiwan,0.576,118
Thailand,0.555,119
Chile,0.550,131
Greece,0.548,252
Spain,0.545,717
Romania,0.529,323
Philippines,0.528,142
Colombia,0.525,202


In [19]:
# Which specific AI models are most common among adopters overall?
# The list column round-trips through CSV as a string repr (e.g. "['a', 'b']"), so parse it back safely.
import itertools, ast
model_lists = df['AIModelsHaveWorkedWith_list'].dropna()
all_model_mentions = list(itertools.chain.from_iterable(
    lst if isinstance(lst, list) else ast.literal_eval(lst) for lst in model_lists
))
pd.Series(all_model_mentions).value_counts(normalize=True).round(3).head(10)

openAI GPT (chatbot models)              0.236
Anthropic: Claude Sonnet                 0.124
Gemini (Flash general purpose models)    0.103
openAI Reasoning models                  0.101
openAI Image generating models           0.077
Gemini (Pro Reasoning models)            0.074
DeepSeek (R- Reasoning models)           0.068
Meta Llama (all models)                  0.052
DeepSeek (V- General purpose models)     0.042
X Grok models                            0.032
Name: proportion, dtype: float64

**Reading:** country-level adoption is read the same way as industry/role relative to the baseline, and only among countries with enough respondents (n≥100) to trust the rate. The model-popularity table shows which specific tools dominate globally; a natural follow-up would be repeating that ranking within just the top 2–3 countries to see whether model preference itself varies geographically.

**Cross-check against teammates' pivot tables:** the three CSVs from teammates (`Industries using AI`, `Positions with AI Usage Trend`, `Trend of AI Usage per Country`) use `COUNTA of AIModelsHaveWorkedWith` grouped the same way, but their tables report **counts of adopters**, not a rate, and appear to be filtered on a different base population than `survey_clean.csv` here (their totals don't sum to 49,191 either). The *ranking* of top industries/roles/countries by adopter count in their tables is directionally consistent with the `adoption_rate` ranking above Software Development, back-end/full-stack roles, and the largest countries by respondent volume all show up at the top of both versions but the exact figures won't match one-to-one because the grouping/filter definitions differ. Worth resolving as a group before the final report: agree on one shared definition of "adopter" and one filter (e.g. include/exclude respondents with missing `Industry`) so every teammate's numbers reconcile.

## Statistical reasoning for the adoption question

showed *descriptive* adoption rates by industry, role, and country. This section tests whether each of those breakdowns is a real association or could plausibly be sampling noise the same chi-square approach used in for the individual research question, applied here to `AI_adopter` against each grouping variable.

In [29]:
from scipy import stats as scipy_stats

def chi_square_report(df, group_col, min_n):
    """Chi-square test of AI_adopter independence from group_col, restricted to groups with >= min_n respondents."""
    counts = df[group_col].value_counts()
    keep = counts[counts >= min_n].index
    sub = df[df[group_col].isin(keep)]
    table = pd.crosstab(sub[group_col], sub['AI_adopter'])
    chi2, p, dof, expected = scipy_stats.chi2_contingency(table)
    return chi2, p, dof, table.shape[0], len(sub)

for col, min_n in [('Industry', 50), ('DevType', 50), ('Country', 100)]:
    chi2, p, dof, n_groups, n_rows = chi_square_report(df, col, min_n)
    print(f"{col:10s}  groups={n_groups:3d}  n={n_rows:6d}  chi2={chi2:8.1f}  dof={dof:3d}  p={p:.2e}")

Industry    groups= 15  n= 33642  chi2=    91.8  dof= 14  p=1.77e-13
DevType     groups= 32  n= 43680  chi2=   737.9  dof= 31  p=3.03e-135
Country     groups= 53  n= 33128  chi2=   244.5  dof= 52  p=1.00e-26


**Plain-language interpretation:** all three tests return p-values far below 0.05, meaning the adoption-rate differences  are very unlikely to be pure sampling noise industry, role, and country are each genuinely associated with whether a respondent reports using an AI model, within this survey's respondent pool.

 three caveats apply equally here:
1. **A tiny p-value is not a large effect.** With sample sizes in the thousands to tens of thousands, even a modest, practically small difference in adoption rate will produce a very low p-value the adoption-rate tables themselves, not the p-value, are what should drive any claim about *how much* a group differs.
2. **Association is not causation.** A country or industry having a higher adoption rate doesn't mean that country or industry *causes* higher AI use respondent mix, available tooling, income level, or the survey's own reach into that population could explain the pattern instead.
3. **This describes the respondent pool, not the general population.** The `Country` test in particular should not be read as "developers in China are more likely to adopt AI" as a population-level claim it describes respondents to this specific self-selected survey.

In [30]:
# 95% CI on one specific, question-relevant proportion, mirroring Section 7's approach:
# adoption rate among the single largest country in the sample, for a concrete anchor number
top_country = df['Country'].value_counts().index[0]
sub = df[df['Country'] == top_country]['AI_adopter'].dropna()
p_hat = sub.mean()
n = len(sub)
se = (p_hat * (1 - p_hat) / n) ** 0.5
ci_low, ci_high = p_hat - 1.96 * se, p_hat + 1.96 * se
print(f"Largest-sample country: {top_country}  (n={n})")
print(f"Adoption rate: {p_hat:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")

Largest-sample country: United States of America  (n=7233)
Adoption rate: 0.433
95% CI: [0.422, 0.445]


**Reading:** this CI is deliberately computed for the country with the most respondents (not the highest rate) so the interval is as tight and trustworthy as possible a good habit to repeat for whichever specific country ends up highlighted in the final report, since a country near the top of the ranking but with only ~100-150 respondents will have a noticeably wider, less certain interval than this one.

## Pitfall check for the adoption question

- **Small-sample overconfidence:** both the industry and country breakdowns applied a minimum respondent count (n≥50 / n≥100) specifically so a group with a handful of responses can't produce a headline-grabbing but meaningless 100% or 0% adoption rate.
- **Cherry-picking:** the full sorted tables are shown, not just the top 3 "best" rows, so a reader can see the whole distribution, including where it's flat and unremarkable.
- **Correlation vs. causation:** an industry or role having a higher adoption rate doesn't mean that industry *causes* higher AI use it may reflect what kind of work is being done, tooling budgets, or the survey's own respondent mix for that industry. Flagged explicitly in Section 12.5 alongside the chi-square results.
- **Statistical backing, not just eyeballed rankings:** runs a chi-square test for each of industry, role, and country against `AI_adopter`, confirming the differences shown in the ranked tables are unlikely to be sampling noise rather than presenting a sorted bar chart as if a visible gap alone were evidence of a real effect.
- **Reconciliation limitation:** as noted in , this notebook's adoption figures and the teammates' pivot-table figures are not expected to match exactly due to differing base populations/filters this is flagged explicitly rather than presented as if the numbers agree.
- **Sampling considerations:** same self-selection caveat as this is a self-selected survey population, and country-level adoption in particular should not be read as representing "developers in that country" generally.

## Summary for the visualization set

Findings from this notebook that a chart should be built around:

**AI-learning path (individual proposal):**
1. AI-learning path (curiosity vs. job-required vs. none) by experience tier table, the core finding.
2. AI-learning path by top developer roles — a secondary/weaker pattern worth showing for contrast.
3. Average tool-endorsement rank by AI-learning path — for the second half of the research question.
4. Distribution of years of coding experience — to justify why tiering (not raw years) was used for the comparisons.

**AI model adoption (group question):**
5. Adoption rate by industry , sorted, with the overall baseline marked as a reference line.
6. Adoption rate by developer role , same treatment.
7. Top AI-adopting countries alongside the most-used AI models overall.
8. Most-common AI models overall a simple ranked bar chart, useful as a scene-setter before the group-by breakdowns.